# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batsykods/FLrank1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2 — Refresh / Content Opportunity Scoring.**

I picked this lane over the other three (ranking-signal analysis, archetype clustering,
CTR/engagement scoring) because it runs the *full* ML loop end to end — a hand-written baseline,
a trained model that has to beat that baseline, a leakage check, honest client-holdout
validation, and a ranked output an editor could actually act on. The other lanes each cover a
slice of that loop; this one covers all of it, and the repo's own scripts (`scripts/01`–`05`) and
skills (`building-baselines`, `training-honest-models`, `hunting-leakage-and-validating`) are
built around exactly this path. It's also the most portfolio-legible story: "a model that beats a
hand-written rule at prioritizing editorial work," with a concrete before/after number to show
for it.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

## 2. The question: decision, action, cost of a wrong call

**Question:** Given a client's existing content, which pages should an editor review first this
cycle — refresh, expand, or leave alone?

**Decision it improves:** the order of the editorial review queue. Right now that queue is either
worked in an arbitrary order (newest first, alphabetical, whatever an editor remembers) or built
from a fixed rule someone wrote once and never revisited.

**Who acts on it:** a content editor or SEO strategist with limited review bandwidth per cycle —
they can realistically get through a few dozen items, not the whole inventory.

**Cost of a wrong call, in both directions:**
- *False positive* (flagged as declining, isn't really): wastes an editor's limited time reviewing
  a page that didn't need it — the review slot that page took is a slot some other page didn't get.
- *False negative* (a genuinely declining page never surfaces): the page keeps losing visibility
  unnoticed until it's found by accident or a client complaint, by which point more traffic has
  already been lost than if it had been caught early.

Neither error is catastrophic on its own — this is a prioritization aid, not an automated
publishing action — but a system that's wrong more often than a human's gut feeling isn't worth
using, which is exactly why the output has to be checked against a fixed-rule baseline, not just
judged in isolation.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [1]:
import os
import numpy as np
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"1) Scale: {len(df):,} content items across {df['client_id'].nunique()} clients "
      f"— far more than any editor can hand-review every cycle.")

print(f"2) Skew: {df['is_declining_label'].mean():.1%} of items are currently declining "
      f"— common enough that ignoring it is costly, not so common that 'assume everything is "
      f"declining' is a usable strategy.")

# a quick, deliberately naive rule: "old and not recently updated" = review it
naive_flag = (df["content_age_days"] >= 180) & (df["days_since_last_update"] >= 90)
naive_first50 = df[naive_flag].head(50)
print(f"3) A naive age-based rule ('old + stale') flags {naive_flag.sum():,} items; "
      f"among the first 50 of those (in raw row order, i.e. no real ranking), only "
      f"{naive_first50['is_declining_label'].mean():.1%} are actually declining — barely above "
      f"a coin flip, and no better than the {df['is_declining_label'].mean():.1%} base rate. Age "
      f"and staleness alone don't sort pages by who's actually declining, which is the whole "
      f"reason this lane is worth doing properly.")

1) Scale: 30,000 content items across 32 clients — far more than any editor can hand-review every cycle.
2) Skew: 54.2% of items are currently declining — common enough that ignoring it is costly, not so common that 'assume everything is declining' is a usable strategy.
3) A naive age-based rule ('old + stale') flags 7,141 items; among the first 50 of those (in raw row order, i.e. no real ranking), only 50.0% are actually declining — barely above a coin flip, and no better than the 54.2% base rate. Age and staleness alone don't sort pages by who's actually declining, which is the whole reason this lane is worth doing properly.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

## 4. Careful words: what I can and can't claim

**What this work can say:**
- The label is an **observed, measured** outcome — a real trailing-window change in traffic
  (`trend_direction`/`trend_pct`), not something invented for this exercise.
- The output is **directional and decision-support**: "these pages are worth reviewing sooner
  than those ones," ranked by likelihood, not a certainty.
- Any pattern I find is a **correlation observed in this sample of 30,000 pseudonymized items
  across 32 clients** — real, but specific to this data and this time window.

**What this work can never say:**
- **Not causal proof.** If "thin word count" correlates with declining, that doesn't mean adding
  words *causes* recovery — plenty of other things move alongside content depth.
- **Not "predicting Google."** I'm not modeling the search engine's ranking algorithm — I'm
  modeling which of *our own* traffic patterns, already measured, tend to co-occur with decline.
- **Not a replacement for editorial judgment.** The ranked queue is a starting point for a human
  to check, not an automatic instruction to change or remove a page.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.